In [1]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv
load_dotenv()

c:\Users\davis\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


True

Multiquery retriever has more coverage or reach over documents as compared to normal vector store retriever because it break ambiguous question (query) into multiple queries.

In [2]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever 

In [3]:
llm  = ChatOpenAI(name="gpt-3.5-turbo")
embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

In [4]:
docs = [
    #Europe
    Document(page_content="Paris is famous for the Eiffel Tower and romantic atmosphere."),
    Document(page_content="Rome is known for the Colosseum and ancient ruins."),
    Document(page_content="Switzerland is popular for its ALps and scenic train rides."),

    #Asia
    Document(page_content="Tokyo is a bustling city known for its technology and culture."),
    Document(page_content="Bali is an Indonesian island famous for its beaches and temples."),
    Document(page_content="Thailand is well-known for its street food and tropical islands."),

    #America
    Document(page_content="New York city is known for Times Square and Central Park."),
    Document(page_content="Brazil is famous for the Amazon rainforest and Rio carnival."),
    Document(page_content="Canada is famous for Niagara Falls and natural landscapes."),

    #Irrelevant
    Document(page_content="The human heart pumps blood throughout the body."),
    Document(page_content="Einstein proposed the theory of relativity."),
    Document(page_content="Python is a programming language widely used in AI."),
    Document(page_content="Coffee is one of the most consumed beverages in the world.")
]

In [5]:
query = "What are the best places to visit?"

In [6]:
vector_store = FAISS.from_documents(documents=docs, embedding=embeddings)

In [7]:
baseRetriever = vector_store.as_retriever()

In [8]:
retriever_multi = MultiQueryRetriever.from_llm(retriever=baseRetriever, llm=llm) #split generic query into specific query

In [9]:
result = baseRetriever.invoke(query)
print(result)

[Document(id='df2de88d-ce4a-42d5-bcbf-5c8afd6303c1', metadata={}, page_content='Switzerland is popular for its ALps and scenic train rides.'), Document(id='dd0a6f09-58b0-414b-9946-ba7019cb02bf', metadata={}, page_content='Canada is famous for Niagara Falls and natural landscapes.'), Document(id='d2a7a67a-f3f3-4fa2-9d70-6c4c37f520d4', metadata={}, page_content='Thailand is well-known for its street food and tropical islands.'), Document(id='c23a1312-005f-4eac-907a-5edbfcb87962', metadata={}, page_content='Rome is known for the Colosseum and ancient ruins.')]


In [10]:
result_multi = retriever_multi.invoke(query)
print(result_multi)

[Document(id='df2de88d-ce4a-42d5-bcbf-5c8afd6303c1', metadata={}, page_content='Switzerland is popular for its ALps and scenic train rides.'), Document(id='dd0a6f09-58b0-414b-9946-ba7019cb02bf', metadata={}, page_content='Canada is famous for Niagara Falls and natural landscapes.'), Document(id='d2a7a67a-f3f3-4fa2-9d70-6c4c37f520d4', metadata={}, page_content='Thailand is well-known for its street food and tropical islands.'), Document(id='b8e5561e-5950-4d53-a634-bd1966da30ac', metadata={}, page_content='Paris is famous for the Eiffel Tower and romantic atmosphere.'), Document(id='c1adb531-35ea-48f7-baee-96bd8cf57df7', metadata={}, page_content='Bali is an Indonesian island famous for its beaches and temples.'), Document(id='61f29976-929e-42d1-a622-85c3af0cd834', metadata={}, page_content='Brazil is famous for the Amazon rainforest and Rio carnival.'), Document(id='c23a1312-005f-4eac-907a-5edbfcb87962', metadata={}, page_content='Rome is known for the Colosseum and ancient ruins.')]


In [11]:
print("Result from Vector Store Retriever")
for i, res in enumerate(result):
    print("Result:", i+1)
    print(res.page_content)

Result from Vector Store Retriever
Result: 1
Switzerland is popular for its ALps and scenic train rides.
Result: 2
Canada is famous for Niagara Falls and natural landscapes.
Result: 3
Thailand is well-known for its street food and tropical islands.
Result: 4
Rome is known for the Colosseum and ancient ruins.


In [12]:
print("Result from Multi Query Retriever")
for i, res in enumerate(result_multi):
    print("Result:", i+1)
    print(res.page_content)

Result from Multi Query Retriever
Result: 1
Switzerland is popular for its ALps and scenic train rides.
Result: 2
Canada is famous for Niagara Falls and natural landscapes.
Result: 3
Thailand is well-known for its street food and tropical islands.
Result: 4
Paris is famous for the Eiffel Tower and romantic atmosphere.
Result: 5
Bali is an Indonesian island famous for its beaches and temples.
Result: 6
Brazil is famous for the Amazon rainforest and Rio carnival.
Result: 7
Rome is known for the Colosseum and ancient ruins.


In [ ]:
expanded_query = retriever_multi.llm_chain.invoke({"question": query})#separate calls as compared to earlier multiquery invoke
print("list of specific queries ....")
print(expanded_query)

list of specific queries ....
['1. Can you recommend top travel destinations worth visiting?', "2. Which are some of the must-see places that I shouldn't miss?", '3. What are some highly recommended tourist spots that I should check out?']


In [14]:
for qn in expanded_query:
    print(qn)

1. Can you recommend top travel destinations worth visiting?
2. Which are some of the must-see places that I shouldn't miss?
3. What are some highly recommended tourist spots that I should check out?
